# Battery pipeline — Colab extract phase

This notebook handles **only the heavy I/O part**: read raw .pkl files from
Google Drive, run audit + feature extraction, and download the resulting CSVs
as a small ZIP.

The CPU-light parts of the pipeline (splits, VIF screening, model training,
ablations) run **locally on your laptop** afterwards — see "Phase B" at the
end of this notebook.

This split exists because the raw data is ~15-20 GB on Drive, but the feature
CSVs the audits produce are only a few hundred KB. Once you have those CSVs
committed to Git, you don't need Drive or Colab again unless you change the
feature definitions.

```
                Phase A (this notebook, Colab)              Phase B (your laptop)
┌──────────────────┐  ┌──────────────────────────────┐  ┌──────────────────────────────────┐
│ Drive: raw .pkls │→ │ audit + feature extraction   │→ │ splits + VIF + experiments       │
│ (15-20 GB)       │  │ ~5 min, produces ~200 KB CSV │  │ Seconds to minutes; iterate freely│
└──────────────────┘  └──────────────────────────────┘  └──────────────────────────────────┘
```

## 0. Configuration

In [ ]:
# Edit these to match your setup if needed.

GITHUB_REPO = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
BRANCH = "main"

# Where the raw data lives in your Drive (used if MOUNT_DRIVE=True).
MATR_DRIVE_DIR = "/content/drive/MyDrive/Braatz_NatEnergy2019"   # contains batch1.pkl, batch2.pkl, batch3.pkl
HUST_DRIVE_DIR = "/content/drive/MyDrive/HUST"                  # contains 1-1.pkl, 1-2.pkl, ..., 10-8.pkl

# If True: mount Drive and symlink the folders above into the repo.
# If False: use gdown to fetch from the public shared folders into Colab disk.
MOUNT_DRIVE = True

# Feature extraction toggles.
EOL_FRACTION = 0.85          # SOP raised from 0.80 → 0.85 so MATR batch1/3 stay modelable
EXTRA_WINDOWS = []           # add 25 to compute features for the N=25 ablation too
CAPACITY_NORMALIZE = False   # SOP §2.3: only flip when adding a third dataset with a different cell

## 1. Clone the repo and install dependencies

In [ ]:
import os, subprocess, shutil
from pathlib import Path

REPO_DIR = Path('/content/Graduation-Project-Dicle')
if REPO_DIR.exists():
    print('[clone] repo already present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print(f'[clone] {GITHUB_REPO} -> {REPO_DIR}')
    subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Lightweight install — only what the audit + feature scripts need.
%pip install -q gdown h5py scipy

## 2. Make raw data available under data/raw/

Two paths — `MOUNT_DRIVE = True` symlinks your already-downloaded MyDrive folders into
the repo (no copy, no extra disk). Otherwise we use gdown against the public shared
folders (Anyone-with-link, Viewer).

In [ ]:
RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
HUST_LINK = RAW_DIR / 'HUST_data'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    matr_src = Path(MATR_DRIVE_DIR)
    hust_src = Path(HUST_DRIVE_DIR)
    assert matr_src.exists(), f'MATR Drive folder not found: {matr_src}'
    assert hust_src.exists(), f'HUST Drive folder not found: {hust_src}'
    # Symlink the three MATR pkls into data/raw/
    for fname in ('batch1.pkl', 'batch2.pkl', 'batch3.pkl'):
        target = matr_src / fname
        link = RAW_DIR / fname
        if link.exists() or link.is_symlink():
            link.unlink()
        if not target.exists():
            print(f'[warn] {target} missing in Drive — skipping')
            continue
        link.symlink_to(target)
        print(f'[link] {link.name}  ->  {target}')
    # Symlink the HUST folder
    if HUST_LINK.exists() or HUST_LINK.is_symlink():
        if HUST_LINK.is_symlink():
            HUST_LINK.unlink()
        else:
            shutil.rmtree(HUST_LINK)
    HUST_LINK.symlink_to(hust_src)
    print(f'[link] {HUST_LINK}  ->  {hust_src}')
else:
    print('[gdown] fetching from public shared folders (~15-20 GB into Colab disk)')
    subprocess.run(['python', '0_data_prep/download_data.py'], check=True)

# Sanity check
for p in [RAW_DIR / 'batch1.pkl', RAW_DIR / 'batch2.pkl', RAW_DIR / 'batch3.pkl', HUST_LINK]:
    print('  exists' if p.exists() else '  MISSING', p)

## 3. Run the extract phase (audit + features)

Three steps:
- `audit_matr` reads `data/raw/batch{1,2,3}.pkl` → `matr_cell_audit_*.csv`, `matr_retention_summary.csv`
- `audit_hust` reads `data/raw/HUST_data/*.pkl` → `hust_cycles_tidy.csv`, `hust_threshold_audit.csv`
- `features` reads both → `features_sop12_{matr,hust,combined}.csv`

The first one is fast, the second is slowest (Coulomb counting over 77 cells), the third is fast.

In [ ]:
# Run audit_matr + audit_hust via the orchestrator
subprocess.run(
    ['python', 'run_pipeline.py', '--skip-download', '--stages', 'audit_matr', 'audit_hust'],
    check=True,
)

In [ ]:
# Build the SOP12 feature table.
windows = sorted(set([50, 100, *EXTRA_WINDOWS]))
cmd = [
    'python', '1_feature_engineering/build_sop12_features_v2.py',
    '--eol-fraction', str(EOL_FRACTION),
    '--n-windows', *[str(w) for w in windows],
]
if CAPACITY_NORMALIZE:
    cmd.append('--capacity-normalize')
print('$', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 4. Pack the extract outputs into a ZIP and download

Bundles only the small artifacts produced by the audit + features stages.
No raw .pkl, no model results — those happen locally in Phase B.

In [ ]:
import datetime, zipfile

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
zip_path = Path('/content') / f'extract_outputs_{stamp}.zip'

# Only the data/intermediate artifacts the audit + features stages produce.
# Splits, VIF report, model results all happen locally in Phase B.
INCLUDE_FILES = [
    'data/intermediate/matr_cell_audit_strict.csv',
    'data/intermediate/matr_cell_audit_replication.csv',
    'data/intermediate/matr_retention_summary.csv',
    'data/intermediate/hust_cycles_tidy.csv',
    'data/intermediate/hust_threshold_audit.csv',
    'data/intermediate/hust_threshold_summary.csv',
    'data/intermediate/features_sop12_matr.csv',
    'data/intermediate/features_sop12_hust.csv',
    'data/intermediate/features_sop12_combined.csv',
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel in INCLUDE_FILES:
        path = REPO_DIR / rel
        if not path.exists():
            print(f'[skip] {rel} (not produced this run)')
            continue
        zf.write(path, rel)
        print(f'  +{rel}  ({path.stat().st_size / 1024:.1f} KB)')

print(f'\n[zip] wrote {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Trigger the browser download.
from google.colab import files
files.download(str(zip_path))

## 5. Phase B — local model training

On your laptop:

```bash
cd /Users/osmancifci/Graduation-Project-Dicle
git pull
unzip -o ~/Downloads/extract_outputs_*.zip
git add data/intermediate
git commit -m 'extract phase: refresh feature CSVs from Colab'
git push

# Now everything downstream is CPU-light. Iterate freely:
python run_pipeline.py --phase model           # splits + VIF report + 6-model experiments
python run_pipeline.py --stages experiments   # only re-run experiments after tweaks

# Or call individual scripts with custom flags:
python 2_modeling_featuring/run_experiments_v2.py --models pls catboost gaussian_process
python 2_modeling_featuring/vif_screening.py --drop
python 2_modeling_featuring/run_experiments_v2.py \
    --features-from data/intermediate/vif_kept_features.txt \
    --output-dir outputs/results_v2_vif_drop
```

Phase B outputs (`splits/sop_v2/`, `outputs/results_v2/`, etc.) live alongside
the Phase A artifacts in the same repo, all small enough to commit to Git.